In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [2]:
# Load cleaned datasets

orders = pd.read_csv("../data/processed/orders_clean.csv")

customers = pd.read_csv("../data/processed/customers_clean.csv")

products = pd.read_csv("../data/processed/products_clean.csv")

sellers = pd.read_csv("../data/processed/sellers_clean.csv")

order_items = pd.read_csv("../data/processed/order_items_clean.csv")

payments = pd.read_csv("../data/processed/payments_clean.csv")

reviews = pd.read_csv("../data/processed/reviews_clean.csv")

category_translation = pd.read_csv(
    "../data/processed/category_translation_clean.csv"
)

In [3]:
datasets = {
    "Orders": orders,
    "Customers": customers,
    "Products": products,
    "Sellers": sellers,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Category Translation": category_translation
}

summary = pd.DataFrame({
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
}, index=datasets.keys())

summary

,Rows,Columns
Orders,99441,13
Customers,99441,5
Products,32340,9
Sellers,3095,4
Order Items,112650,7
Payments,103886,5
Reviews,99224,7
Category Translation,71,2


In [4]:
print("Orders")
print(orders.columns.tolist())

print("\nCustomers")
print(customers.columns.tolist())

print("\nOrder Items")
print(order_items.columns.tolist())

print("\nProducts")
print(products.columns.tolist())

print("\nSellers")
print(sellers.columns.tolist())

print("\nPayments")
print(payments.columns.tolist())

print("\nReviews")
print(reviews.columns.tolist())

print("\nCategory Translation")
print(category_translation.columns.tolist())

Orders
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_weekday', 'delivery_days']

Customers
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Order Items
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Products
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Sellers
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Payments
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Reviews
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_mes

In [5]:
print("Orders:", orders["order_id"].is_unique)

print("Customers:", customers["customer_id"].is_unique)

print("Products:", products["product_id"].is_unique)

print("Sellers:", sellers["seller_id"].is_unique)

Orders: True
Customers: True
Products: True
Sellers: True


In [6]:
print("Unique Orders in Order Items:",
      order_items["order_id"].nunique())

print("Rows in Order Items:",
      len(order_items))

print()

print("Unique Orders in Payments:",
      payments["order_id"].nunique())

print("Rows in Payments:",
      len(payments))

print()

print("Unique Orders in Reviews:",
      reviews["order_id"].nunique())

print("Rows in Reviews:",
      len(reviews))

Unique Orders in Order Items: 98666
Rows in Order Items: 112650

Unique Orders in Payments: 99440
Rows in Payments: 103886

Unique Orders in Reviews: 98673
Rows in Reviews: 99224


In [7]:
# Create the initial master dataset
master_df = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", master_df.shape)

print("\nUnique Orders:", master_df["order_id"].nunique())

print("Missing Customer IDs:", master_df["customer_id"].isna().sum())

Shape: (99441, 17)

Unique Orders: 99441
Missing Customer IDs: 0


In [8]:
# Merge Orders + Customers with Order Items

master_df = master_df.merge(
    order_items,
    on="order_id",
    how="left",
    validate="one_to_many"
)

print("Shape:", master_df.shape)

print("\nUnique Orders:", master_df["order_id"].nunique())

print("Missing Product IDs:", master_df["product_id"].isna().sum())

print("Average Items Per Order:",
      round(len(master_df) / master_df["order_id"].nunique(), 2))

Shape: (113425, 23)

Unique Orders: 99441
Missing Product IDs: 775
Average Items Per Order: 1.14


In [9]:
# Merge product information

master_df = master_df.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", master_df.shape)

print("\nUnique Orders:", master_df["order_id"].nunique())

print("Missing Product Categories:",
      master_df["product_category_name"].isna().sum())

print("Missing Product IDs:",
      master_df["product_id"].isna().sum())

Shape: (113425, 31)

Unique Orders: 99441
Missing Product Categories: 2379
Missing Product IDs: 775


In [10]:
# Merge category translation

master_df = master_df.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

print("Shape:", master_df.shape)

print("\nUnique Orders:", master_df["order_id"].nunique())

print("Missing English Categories:",
      master_df["product_category_name_english"].isna().sum())

print("\nSample Categories:")

master_df[
    [
        "product_category_name",
        "product_category_name_english"
    ]
].drop_duplicates().head(10)

Shape: (113425, 32)

Unique Orders: 99441
Missing English Categories: 2403

Sample Categories:


,product_category_name,product_category_name_english
0,utilidades_domesticas,housewares
1,perfumaria,perfumery
2,automotivo,auto
3,pet_shop,pet_shop
4,papelaria,stationery
6,NaN,NaN
8,moveis_decoracao,furniture_decor
9,moveis_escritorio,office_furniture
10,ferramentas_jardim,garden_tools
12,informatica_acessorios,computers_accessories


In [11]:
# Merge seller information

master_df = master_df.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", master_df.shape)

print("\nUnique Orders:", master_df["order_id"].nunique())

print("Missing Seller IDs:",
      master_df["seller_id"].isna().sum())

print("Missing Seller State:",
      master_df["seller_state"].isna().sum())

print("\nSeller States:")

print(
    master_df["seller_state"]
    .value_counts(dropna=False)
    .head(10)
)

Shape: (113425, 35)

Unique Orders: 99441
Missing Seller IDs: 775
Missing Seller State: 775

Seller States:
seller_state
SP     80342
MG      8827
PR      8671
RJ      4818
SC      4075
RS      2199
DF       899
NaN      775
BA       643
GO       520
Name: count, dtype: int64


## Merge 5: Sellers

**Purpose:**
Enrich the master dataset with seller location information.

**Join Key:**
- seller_id

**Join Type:**
- LEFT JOIN

**Validation:**
- many_to_one

**Expected Result:**
- Row count remains unchanged.
- Seller location columns are added.

In [12]:
payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_type=("payment_type", lambda x: ", ".join(sorted(x.unique()))),
        payment_count=("payment_sequential", "count")
    )
    .reset_index()
)

payment_summary.head()

,order_id,payment_value,payment_installments,payment_type,payment_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card,1


In [13]:
print("Rows:", payment_summary.shape[0])

print("Unique Orders:",
      payment_summary["order_id"].nunique())

print(payment_summary.head())

Rows: 99440
Unique Orders: 99440
                           order_id  payment_value  payment_installments  \
0  00010242fe8c5a6d1ba2dd792cb16214          72.19                     2   
1  00018f77f2f0320c557190d7a144bdd3         259.83                     3   
2  000229ec398224ef6ca0657da4fc703e         216.87                     5   
3  00024acbcdf0a6daa1e931b038114c75          25.78                     2   
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04                     3   

  payment_type  payment_count  
0  credit_card              1  
1  credit_card              1  
2  credit_card              1  
3  credit_card              1  
4  credit_card              1  


In [14]:
master_df = master_df.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", master_df.shape)

print("\nUnique Orders:",
      master_df["order_id"].nunique())

print("Missing Payment Value:",
      master_df["payment_value"].isna().sum())

print("Missing Payment Type:",
      master_df["payment_type"].isna().sum())

Shape: (113425, 39)

Unique Orders: 99441
Missing Payment Value: 3
Missing Payment Type: 3


In [15]:
review_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count"),
        review_comment_title=("review_comment_title", "first"),
        review_comment_message=("review_comment_message", "first")
    )
    .reset_index()
)

review_summary.head()

,order_id,review_score,review_count,review_comment_title,review_comment_message
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1,None,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,4.0,1,None,None
2,000229ec398224ef6ca0657da4fc703e,5.0,1,None,Chegou antes do prazo previsto e o produto surpreendeu pela qualidade. Muito satisfatório.
3,00024acbcdf0a6daa1e931b038114c75,4.0,1,None,None
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1,None,Gostei pois veio no prazo determinado .


In [16]:
print("Rows:", review_summary.shape[0])

print("Unique Orders:",
      review_summary["order_id"].nunique())

Rows: 98673
Unique Orders: 98673


In [17]:
master_df = master_df.merge(
    review_summary,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", master_df.shape)

print("\nUnique Orders:",
      master_df["order_id"].nunique())

print("Missing Review Score:",
      master_df["review_score"].isna().sum())

print("Missing Review Count:",
      master_df["review_count"].isna().sum())

Shape: (113425, 43)

Unique Orders: 99441
Missing Review Score: 961
Missing Review Count: 961


In [18]:
print("=" * 60)
print("MASTER DATASET SUMMARY")
print("=" * 60)

print(f"Rows: {master_df.shape[0]:,}")
print(f"Columns: {master_df.shape[1]}")

print(f"\nUnique Orders: {master_df['order_id'].nunique():,}")
print(f"Unique Customers: {master_df['customer_id'].nunique():,}")
print(f"Unique Products: {master_df['product_id'].nunique():,}")
print(f"Unique Sellers: {master_df['seller_id'].nunique():,}")

print("\nMissing Values:")

print(master_df.isnull().sum().sort_values(ascending=False).head(15))

MASTER DATASET SUMMARY
Rows: 113,425
Columns: 43

Unique Orders: 99,441
Unique Customers: 99,441
Unique Products: 32,951
Unique Sellers: 3,095

Missing Values:
review_comment_title             99920
review_comment_message           65458
order_delivered_customer_date     3229
delivery_days                     3229
product_category_name_english     2403
product_width_cm                  2379
product_height_cm                 2379
product_length_cm                 2379
product_weight_g                  2379
product_photos_qty                2379
product_description_lenght        2379
product_name_lenght               2379
product_category_name             2379
order_delivered_carrier_date      1968
review_score                       961
dtype: int64


In [19]:
master_df["product_id"].nunique(dropna=False)

32952

In [22]:
master_df["product_id"].nunique()

32951

In [25]:
# Product IDs present in order_items but missing from products

missing_products = (
    set(order_items["product_id"].dropna())
    - set(products["product_id"].dropna())
)

print("Missing product IDs:", len(missing_products))

list(missing_products)[:10]

Missing product IDs: 611


['c9e0665ea8ce2a04e4db1b53be9fc0d2',
 '3a78f64aac654298e4b9aff32fc21818',
 '0fe09f32709c79ba4bf19d43c4e49515',
 'bc107cfd28a282c85b28e7054aeb729d',
 'b1d207586fca400a2370d50a9ba1da98',
 'e3c816666a7d2a1e7fbf02651e550b78',
 '3f602d35212cda35342fc92d0ebc32e1',
 'cebad0ed16ecd450b97d2be843d3da86',
 '41eee23c25f7a574dfaf8d5c151dbb12',
 '5c9ef6c35fdbad9275157b1929c37fb0']

In [28]:
order_items[
    order_items["product_id"].isin(missing_products)
].shape

(1604, 7)

In [31]:
# Save the final master dataset

master_df.to_csv(
    "../data/processed/master_sales_dataset.csv",
    index=False
)

print("✅ Master dataset saved successfully!")

✅ Master dataset saved successfully!


In [32]:
master_sales = pd.read_csv("../data/processed/master_sales_dataset.csv")

In [33]:
print("=" * 70)
print("MASTER SALES DATASET VALIDATION")
print("=" * 70)

print(f"Rows              : {master_sales.shape[0]:,}")
print(f"Columns           : {master_sales.shape[1]}")

print("\nUnique Values")
print("-" * 70)

print(f"Orders            : {master_sales['order_id'].nunique():,}")
print(f"Customers         : {master_sales['customer_id'].nunique():,}")
print(f"Products          : {master_sales['product_id'].nunique(dropna=True):,}")
print(f"Sellers           : {master_sales['seller_id'].nunique():,}")

print("\nMissing Values")
print("-" * 70)

print(master_sales.isnull().sum().sort_values(ascending=False).head(15))

MASTER SALES DATASET VALIDATION
Rows              : 113,425
Columns           : 43

Unique Values
----------------------------------------------------------------------
Orders            : 99,441
Customers         : 99,441
Products          : 32,951
Sellers           : 3,095

Missing Values
----------------------------------------------------------------------
review_comment_title             99920
review_comment_message           65458
order_delivered_customer_date     3229
delivery_days                     3229
product_category_name_english     2403
product_width_cm                  2379
product_height_cm                 2379
product_length_cm                 2379
product_weight_g                  2379
product_photos_qty                2379
product_description_lenght        2379
product_name_lenght               2379
product_category_name             2379
order_delivered_carrier_date      1968
review_score                       961
dtype: int64


In [34]:
print("=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

print("Duplicate Rows:",
      master_sales.duplicated().sum())

DUPLICATE CHECK
Duplicate Rows: 0


In [35]:
print("=" * 70)
print("DATA TYPES")
print("=" * 70)

print(master_sales.dtypes)

DATA TYPES
order_id                          object
customer_id                       object
order_status                      object
order_purchase_timestamp          object
order_approved_at                 object
order_delivered_carrier_date      object
order_delivered_customer_date     object
order_estimated_delivery_date     object
purchase_year                      int64
purchase_month                     int64
purchase_day                       int64
purchase_weekday                  object
delivery_days                    float64
customer_unique_id                object
customer_zip_code_prefix           int64
customer_city                     object
customer_state                    object
order_item_id                    float64
product_id                        object
seller_id                         object
shipping_limit_date               object
price                            float64
freight_value                    float64
product_category_name             object
produ

In [36]:
print("=" * 70)
print("MEMORY USAGE")
print("=" * 70)

memory_mb = master_sales.memory_usage(deep=True).sum() / 1024**2

print(f"Dataset Size: {memory_mb:.2f} MB")

MEMORY USAGE
Dataset Size: 190.60 MB


In [37]:
master_sales.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,purchase_year,purchase_month,purchase_day,purchase_weekday,delivery_days,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,payment_count,review_score,review_count,review_comment_title,review_comment_message
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,2,Monday,8.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,38.71,1.0,"credit_card, voucher",3.0,4.0,1.0,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente."
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,24,Tuesday,13.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,141.46,1.0,boleto,1.0,4.0,1.0,Muito boa a loja,Muito bom o produto.
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,8,Wednesday,9.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,179.12,3.0,credit_card,1.0,5.0,1.0,NaN,NaN
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,18,Saturday,13.0,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842.0,belo horizonte,MG,72.20,1.0,credit_card,1.0,5.0,1.0,NaN,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,13,Tuesday,2.0,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752.0,mogi das cruzes,SP,28.62,1.0,credit_card,1.0,5.0,1.0,NaN,NaN


# ETL Validation Summary

## Dataset Summary

- Rows: 113,425
- Columns: 43
- Unique Orders: 99,441
- Unique Customers: 99,441
- Unique Products: 32,951
- Unique Sellers: 3,095

## Merge Validation

| Merge | Validation | Status |
|--------|------------|--------|
| Orders + Customers | many_to_one | ✅ |
| Orders + Order Items | one_to_many | ✅ |
| Order Items + Products | many_to_one | ✅ |
| Products + Category Translation | many_to_one | ✅ |
| Order Items + Sellers | many_to_one | ✅ |
| Orders + Payment Summary | many_to_one | ✅ |
| Orders + Review Summary | many_to_one | ✅ |

## Data Quality Findings

- 775 orders have no matching order items.
- 611 product IDs in `order_items` do not exist in the `products` table (1,604 affected rows).
- 3 orders have no payment records.
- Some orders have no reviews.
- Some products have missing category and dimension information.

## Conclusion

A production-style ETL pipeline was successfully built by integrating multiple normalized tables into a single analytics-ready master dataset. Validation checks confirmed that the joins behaved as expected, while documented data quality issues were identified as originating from the source data rather than the ETL process.

In [38]:
import os

os.listdir("../data/processed")

['reviews_clean.csv',
 'sellers_clean.csv',
 'geolocation_clean.csv',
 'orders_clean.csv',
 'payments_clean.csv',
 'category_translation_clean.csv',
 'customers_clean.csv',
 'order_items_clean.csv',
 'master_sales_dataset.csv',
 'products_clean.csv']